In [7]:
import pandas as pd
import numpy as np
import os

# Load the data
input_path = '../data/2026_meta_info.csv'
df = pd.read_csv(input_path)

# --- Configuration ---
strain_col = 'strain'
mic_cols = ['itc', 'pos', 'vrc'] 
# ---------------------

# 1. Initialize the master dataframe with PLINK ID columns
final_pheno = pd.DataFrame()
final_pheno['FID'] = 0
final_pheno['IID'] = df[strain_col]

# 2. Loop through each drug and add the Log2 column
for drug in mic_cols:
    # Convert to numeric (handles 'NA' strings)
    numeric_mic = pd.to_numeric(df[drug], errors='coerce')
    
    # Calculate Log2 (using a mask to avoid log(0) errors if any MICs are 0)
    final_pheno[f'Log2_{drug}'] = np.log2(numeric_mic.replace(0, np.nan))

# 3. Handle Missing Data
# Replacing NaNs with -9 (Standard PLINK missing value indicator)
final_pheno = final_pheno.fillna(0)

# 4. Save to one single file
output_path = '../results/all_mic_phenotypes.txt'

# Ensure the results directory exists
os.makedirs('../results', exist_ok=True)

final_pheno.to_csv(output_path, sep='\t', index=False, quoting=3)

print(f"Successfully created {output_path} with traits: {', '.join(mic_cols)}")
print(final_pheno.head())

Successfully created ../results/all_mic_phenotypes.txt with traits: itc, pos, vrc
   FID      IID  Log2_itc  Log2_pos  Log2_vrc
0  0.0  V273-64       0.0      -2.0       0.0
1  0.0  M039-56       1.0      -1.0       1.0
2  0.0  M039-58       0.0       1.0      -2.0
3  0.0  V080-81       0.0      -2.0       0.0
4  0.0  V072-57       5.0       1.0      -2.0


The below script converts a nwk file to matrix format 

In [4]:
from ete3 import Tree
import pandas as pd

t = Tree("../data/tree_data/asptu.raxml.tree", format=1)
samples = [leaf.name for leaf in t.iter_leaves()]
distances = pd.DataFrame(index=samples, columns=samples)

for i in samples:
    for j in samples:
        try:
            node_i = t.search_nodes(name=i)[0]
            node_j = t.search_nodes(name=j)[0]

            distances.loc[i, j] = t.get_distance(node_i, node_j)
        except IndexError:
            print(f"Missing node: {i} or {j}")
            distances.loc[i, j] = float('nan')

distances.to_csv("../results/species_distances.tsv", sep="\t")

Now we convert the distances to a PCA covariates format to be used in plink:

In [5]:
import pandas as pd
from sklearn.decomposition import PCA

dist = pd.read_csv("../results/species_distances.tsv", sep="\t", index_col=0)
pca = PCA(n_components=10)
pcs = pca.fit_transform(dist)

# Convert to PLINK covariate format
# PLINK expects: FID, IID, PC1, PC2, ...
pca_df = pd.DataFrame(pcs, columns=[f'PC{i+1}' for i in range(10)])
pca_df.insert(0, 'IID', dist.index)
pca_df.insert(0, 'FID', 0)

# Save for PLINK
pca_df.to_csv("../results/pca_covariates.txt", sep="\t", index=False)